# 🚢 Titanic - Machine Learning from Disaster

## Phase 4: Feature Engineering

### Objective

The objective of this notebook is to transform the existing Titanic passenger information into features that may provide stronger predictive signals for machine learning models.

The features created in this notebook are based on hypotheses generated during Exploratory Data Analysis (EDA).

We will engineer features from:

* Passenger names and titles.
* Family relationships.
* Cabin information.
* Ticket information.
* Passenger fares.
* Passenger age.
* Selected interaction effects

## 1. Import Libraries

We begin by importing the libraries required for feature engineering and data validation.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load the Cleaned Datasets

We will use the minimally cleaned datasets produced during Phase 2 rather than returning to the original raw files.


In [3]:
train = pd.read_csv(
    "/mnt/d/Acelin/Work/Code/titanic-kaggle/data/processed/train_clean.csv"
)

test = pd.read_csv(
    "/mnt/d/Acelin/Work/Code/titanic-kaggle/data/processed/test_clean.csv"
)

train_fe = train.copy()
test_fe = test.copy()

print("Datasets loaded successfully.")

print(
    f"Training shape: {train_fe.shape}"
)

print(
    f"Test shape: {test_fe.shape}"
)

Datasets loaded successfully.
Training shape: (891, 12)
Test shape: (418, 11)


## 3. Initial Validation

Before engineering features, we verify that:

* The training dataset contains the target variable.
* The test dataset does not contain the target.
* `PassengerId` remains unique.
* Train and test contain the same predictor columns.

These checks ensure that feature engineering begins from a consistent dataset state.

In [4]:
assert "Survived" in train_fe.columns
assert "Survived" not in test_fe.columns

assert train_fe["PassengerId"].is_unique
assert test_fe["PassengerId"].is_unique

train_predictors = set(
    train_fe.columns
) - {"Survived"}

test_predictors = set(
    test_fe.columns
)

assert train_predictors == test_predictors

print(
    "Initial validation checks passed."
)

Initial validation checks passed.


## 4. Prepare Data for Consistent Feature Engineering

Some engineered features depend on relationships between passengers.

For example, passengers sharing the same ticket may appear across the competition's training and test datasets.

If ticket-group sizes were calculated separately for training and test data, the same ticket could receive different group-size values depending on which dataset a passenger appears in.

To avoid this inconsistency, we will temporarily combine the predictor columns from both datasets.

### Important Leakage Note

The target variable `Survived` will **not** be included in the combined feature-engineering dataset.

Therefore, no survival information from the training data can be used to construct test-set features.

The combined dataset will only be used for deterministic, target-independent transformations and group counts.


In [5]:
# Preserve the target separately
y = train_fe["Survived"].copy()

# Remove target before combining
train_features = (
    train_fe
    .drop(columns="Survived")
    .copy()
)

test_features = test_fe.copy()

# Mark dataset origin
train_features["_Dataset"] = "train"
test_features["_Dataset"] = "test"

# Combine predictor data
combined = pd.concat(
    [
        train_features,
        test_features
    ],
    axis=0,
    ignore_index=True
)

print(
    f"Combined shape: {combined.shape}"
)

print(
    combined["_Dataset"]
    .value_counts()
)

Combined shape: (1309, 12)
_Dataset
train    891
test     418
Name: count, dtype: int64


## 5. Name-Based Features

Passenger names contain structured information that is not immediately available as a separate column.

A typical Titanic passenger name follows a pattern similar to:

`Surname, Title. First Name`

In [6]:
combined["Title"] = (
    combined["Name"]
    .str.extract(
        r",\s*([^.]*)\.",
        expand=False
    )
    .str.strip()
)

print(
    combined["Title"]
    .value_counts()
)

Title
Mr              757
Miss            260
Mrs             197
Master           61
Rev               8
Dr                8
Col               4
Major             2
Mlle              2
Ms                2
Mme               1
Don               1
Sir               1
Lady              1
Capt              1
the Countess      1
Jonkheer          1
Dona              1
Name: count, dtype: int64


### 5.1 Simplify Passenger Titles

Some titles occur only a small number of times.

Keeping every rare title as a separate category may introduce unnecessary complexity and create categories with insufficient observations.

We will first normalize historically equivalent titles and then group remaining uncommon titles into a `Rare` category.

Common titles such as `Mr`, `Mrs`, `Miss`, and `Master` will remain separate because they may capture meaningful differences in age, sex, and social role.


In [8]:
title_replacements = {
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
}

combined["Title"] = (
    combined["Title"]
    .replace(title_replacements)
)

common_titles = [
    "Mr",
    "Mrs",
    "Miss",
    "Master"
]

combined["Title"] = (
    combined["Title"]
    .where(
        combined["Title"].isin(
            common_titles
        ),
        "Rare"
    )
)

print(
    combined["Title"]
    .value_counts()
)

Title
Mr        757
Miss      264
Mrs       198
Master     61
Rare       29
Name: count, dtype: int64


In [9]:
print(
    pd.crosstab(
        combined["Title"],
        combined["Sex"]
    )
)

Sex     female  male
Title               
Master       0    61
Miss       264     0
Mr           0   757
Mrs        198     0
Rare         4    25


### Feature Created: `Title`

The original high-cardinality `Name` feature has been transformed into a lower-cardinality `Title` feature.

This feature may capture useful information about:

* Passenger sex.
* Approximate age.
* Marital status.
* Social role.

## 6. Family-Based Features

The original dataset contains:

* `SibSp` — Number of siblings or spouses aboard.
* `Parch` — Number of parents or children aboard.

EDA suggested that considering these variables together may provide more useful information about a passenger's travel situation.

We will create:

### `FamilySize`

Total number of immediate family members represented by the available family variables, including the passenger.

`FamilySize = SibSp + Parch + 1`

### `IsAlone`

Binary indicator identifying passengers travelling without immediate family according to `SibSp` and `Parch`.

### `FamilyCategory`

A categorical representation of family size designed to capture potential nonlinear relationships.

These categories remain hypotheses and will later be evaluated using cross-validation.


In [10]:
combined["FamilySize"] = (
    combined["SibSp"] +
    combined["Parch"] +
    1
)

combined["IsAlone"] = (
    combined["FamilySize"] == 1
).astype(int)

print(
    combined["FamilySize"]
    .value_counts()
    .sort_index()
)

print(
    "\nIsAlone distribution:"
)

print(
    combined["IsAlone"]
    .value_counts()
)

FamilySize
1     790
2     235
3     159
4      43
5      22
6      25
7      16
8       8
11     11
Name: count, dtype: int64

IsAlone distribution:
IsAlone
1    790
0    519
Name: count, dtype: int64


In [11]:
def categorize_family_size(size):
    if size == 1:
        return "Alone"

    elif size <= 4:
        return "Small"

    elif size <= 7:
        return "Medium"

    else:
        return "Large"


combined["FamilyCategory"] = (
    combined["FamilySize"]
    .apply(
        categorize_family_size
    )
)

print(
    combined["FamilyCategory"]
    .value_counts()
)

FamilyCategory
Alone     790
Small     437
Medium     63
Large      19
Name: count, dtype: int64


### Features Created

* `FamilySize` — Total immediate family size including the passenger.
* `IsAlone` — Indicates whether the passenger was travelling without immediate family.
* `FamilyCategory` — Groups family sizes into broader categories.

We deliberately avoid creating an arbitrary numerical `Family_Group_ID`.

An identifier assigned to each surname or family group has no natural numerical meaning and may cause some models to interpret unrelated group IDs as ordered quantities.

More advanced family-group features can be explored later using carefully validated approaches.

## 7. Cabin-Based Features

The original `Cabin` column contains substantial missing information.

Rather than dropping the column entirely, we will extract two simpler features.

### `CabinKnown`

Indicates whether cabin information is available.

### `Deck`

Extracts the first letter of the recorded cabin number, which approximately represents the passenger's deck.

Passengers without recorded cabin information will receive the category `Unknown`.

These features preserve potentially useful cabin information while avoiding direct use of the high-cardinality raw cabin strings.


In [12]:
combined["CabinKnown"] = (
    combined["Cabin"]
    .notna()
    .astype(int)
)

combined["Deck"] = (
    combined["Cabin"]
    .fillna("Unknown")
    .astype(str)
    .str[0]
)

combined.loc[
    combined["Cabin"].isna(),
    "Deck"
] = "Unknown"

print(
    combined["Deck"]
    .value_counts()
)

print(
    "\nCabinKnown:"
)

print(
    combined["CabinKnown"]
    .value_counts()
)

Deck
Unknown    1014
C            94
B            65
D            46
E            41
A            22
F            21
G             5
T             1
Name: count, dtype: int64

CabinKnown:
CabinKnown
0    1014
1     295
Name: count, dtype: int64


In [13]:
print(
    pd.crosstab(
        combined["Deck"],
        combined["Pclass"]
    )
)

Pclass    1    2    3
Deck                 
A        22    0    0
B        65    0    0
C        94    0    0
D        40    6    0
E        34    4    3
F         0   13    8
G         0    0    5
T         1    0    0
Unknown  67  254  693


### Features Created

* `CabinKnown` — Indicates whether cabin information was recorded.
* `Deck` — Extracts the first cabin letter or assigns `Unknown`.

Deck and passenger class may contain overlapping information.

Both features will initially be retained so that cross-validation can determine whether deck information provides predictive value beyond `Pclass`.


## 8. Ticket-Based Features

The raw `Ticket` column has high cardinality and is not suitable for direct use by most standard machine learning models.

However, ticket information may reveal:

* Shared bookings.
* Travelling groups.
* Ticket series or booking types.

We will engineer two features:

### `TicketGroupSize`

The total number of passengers in the combined competition data sharing the same ticket number.

### `TicketPrefix`

A normalized alphabetic prefix extracted from the ticket.

Purely numerical tickets will receive the category `NUMERIC`.

In [14]:
ticket_counts = (
    combined["Ticket"]
    .value_counts()
)

combined["TicketGroupSize"] = (
    combined["Ticket"]
    .map(ticket_counts)
)

print(
    combined["TicketGroupSize"]
    .describe()
)

count    1309.000000
mean        2.101604
std         1.779832
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max        11.000000
Name: TicketGroupSize, dtype: float64


In [15]:
def extract_ticket_prefix(ticket):
    ticket = str(ticket).upper().strip()

    # Remove final numeric component
    prefix = (
        ticket
        .rsplit(" ", 1)[0]
        if " " in ticket
        else ""
    )

    # Handle tickets containing only numbers
    if ticket.replace(" ", "").isdigit():
        return "NUMERIC"

    # Keep alphabetic characters only
    prefix = "".join(
        char
        for char in prefix
        if char.isalpha()
    )

    if prefix == "":
        prefix = "".join(
            char
            for char in ticket
            if char.isalpha()
        )

    return (
        prefix
        if prefix
        else "NUMERIC"
    )


combined["TicketPrefix"] = (
    combined["Ticket"]
    .apply(
        extract_ticket_prefix
    )
)

print(
    combined["TicketPrefix"]
    .value_counts()
)

TicketPrefix
NUMERIC      957
PC            92
CA            68
A             39
SOTONOQ       24
STONO         21
SCPARIS       19
WC            15
FCC            9
SOC            8
C              8
SOPP           7
LINE           4
SCAH           4
WEP            4
PP             4
FC             3
SOTONO         3
SCA            3
SC             2
PPP            2
AQ             2
SWPP           2
SP             1
FA             1
SOP            1
AS             1
SCOW           1
SCAHBASLE      1
CASOTON        1
STONOQ         1
LP             1
Name: count, dtype: int64


### 8.1 Reduce Rare Ticket Prefix Categories

Some ticket prefixes occur only a handful of times.

Very rare categories may add unnecessary dimensionality after one-hot encoding.

We will group prefixes with very low frequency into a common `RARE` category.

The threshold used here is a feature-engineering hypothesis rather than a guaranteed optimal value. Its usefulness can later be evaluated through model validation.


In [16]:
ticket_prefix_counts = (
    combined["TicketPrefix"]
    .value_counts()
)

rare_ticket_prefixes = (
    ticket_prefix_counts[
        ticket_prefix_counts < 10
    ]
    .index
)

combined["TicketPrefix"] = (
    combined["TicketPrefix"]
    .replace(
        rare_ticket_prefixes,
        "RARE"
    )
)

print(
    combined["TicketPrefix"]
    .value_counts()
)

TicketPrefix
NUMERIC    957
PC          92
RARE        74
CA          68
A           39
SOTONOQ     24
STONO       21
SCPARIS     19
WC          15
Name: count, dtype: int64


## 9. Fare-Based Features

EDA showed that `Fare` is strongly right-skewed and may contain useful information related to passenger class and travel groups.

We will create two additional fare representations.

### `LogFare`

A logarithmic transformation:

`LogFare = log(1 + Fare)`

This reduces the influence of extremely large fare values and may be useful for linear models.

### `FarePerPerson`

Passengers sharing the same ticket may have a fare value associated with a group booking.

We will therefore calculate:

`FarePerPerson = Fare / TicketGroupSize`


In [17]:
combined["LogFare"] = np.log1p(
    combined["Fare"]
)

combined["FarePerPerson"] = (
    combined["Fare"] /
    combined["TicketGroupSize"]
)

print(
    combined[
        [
            "Fare",
            "LogFare",
            "FarePerPerson"
        ]
    ]
    .describe()
    .round(2)
)

          Fare  LogFare  FarePerPerson
count  1309.00  1309.00        1309.00
mean     33.28     2.98          14.75
std      51.74     0.97          13.55
min       0.00     0.00           0.00
25%       7.90     2.19           7.55
50%      14.45     2.74           8.05
75%      31.28     3.47          15.00
max     512.33     6.24         128.08


In [18]:
assert np.isfinite(
    combined["LogFare"]
).all()

assert np.isfinite(
    combined["FarePerPerson"]
).all()

print(
    "Fare features contain "
    "no infinite values."
)

Fare features contain no infinite values.


## 10. Age-Based Features and Imputation

`Age` contains substantial missing information.

Using the overall mean or median would ignore useful passenger characteristics.

The extracted `Title` feature provides information related to age. Passenger class may also capture differences in passenger demographics.

We will therefore impute missing ages using the median age of passengers with the same:

* `Title`
* `Pclass`

### Leakage Consideration

The imputation statistics are calculated without using `Survived`.

However, in a fully production-grade machine learning workflow, learned imputation statistics are ideally fitted within each training fold during cross-validation.

For this competition notebook, we create a consistent target-independent feature dataset here. Later, we can compare this approach with pipeline-based imputation if necessary.

We will also create an `AgeMissing` indicator before imputation so that the model can distinguish passengers whose original age was unavailable.


In [20]:
combined["AgeMissing"] = (
    combined["Age"]
    .isna()
    .astype(int)
)

print(
    "Missing Age values before imputation:",
    combined["Age"].isna().sum()
)

print(
    "\nAgeMissing distribution:"
)

print(
    combined["AgeMissing"]
    .value_counts()
)

Missing Age values before imputation: 263

AgeMissing distribution:
AgeMissing
0    1046
1     263
Name: count, dtype: int64


In [21]:
age_medians = (
    combined
    .groupby(
        ["Title", "Pclass"]
    )["Age"]
    .median()
)

age_medians

Title   Pclass
Master  1          6.0
        2          2.0
        3          6.0
Miss    1         30.0
        2         20.0
        3         18.0
Mr      1         41.5
        2         30.0
        3         26.0
Mrs     1         45.0
        2         30.5
        3         31.0
Rare    1         48.5
        2         41.5
Name: Age, dtype: float64

In [22]:
def impute_age(row):
    if pd.notna(row["Age"]):
        return row["Age"]

    group_key = (
        row["Title"],
        row["Pclass"]
    )

    group_median = (
        age_medians.get(
            group_key,
            np.nan
        )
    )

    if pd.notna(group_median):
        return group_median

    return combined["Age"].median()


combined["Age"] = combined.apply(
    impute_age,
    axis=1
)

print(
    "Missing Age values after imputation:",
    combined["Age"].isna().sum()
)

Missing Age values after imputation: 0


In [23]:
assert combined["Age"].isna().sum() == 0

print(
    "Age imputation completed successfully."
)

print(
    combined["Age"]
    .describe()
    .round(2)
)

Age imputation completed successfully.
count    1309.00
mean       29.27
std        13.45
min         0.17
25%        21.00
50%        26.00
75%        36.50
max        80.00
Name: Age, dtype: float64


### 10.1 Create Child Indicator

EDA suggested that younger passengers may have survival patterns that differ from those of adults.

We will create an `IsChild` indicator using an age threshold of 12 years.

The original continuous `Age` feature will also be retained.

This allows models to use both:

* Fine-grained continuous age information.
* A simple nonlinear distinction between children and older passengers.

The threshold is a modeling hypothesis and should ultimately be evaluated through validation.


In [24]:
combined["IsChild"] = (
    combined["Age"] < 12
).astype(int)

print(
    combined["IsChild"]
    .value_counts()
)

IsChild
0    1210
1      99
Name: count, dtype: int64


### 10.2 Create Age Groups

Continuous age may have a nonlinear relationship with survival.

We will therefore create broad age categories while retaining the original continuous `Age` variable.

The age groups provide an alternative representation that may be particularly useful for linear models.

Tree-based models may already learn age thresholds automatically, so the usefulness of this categorical representation will be evaluated during model comparison.


In [25]:
combined["AgeGroup"] = pd.cut(
    combined["Age"],
    bins=[
        0,
        12,
        18,
        30,
        50,
        65,
        np.inf
    ],
    labels=[
        "Child",
        "Teen",
        "YoungAdult",
        "Adult",
        "OlderAdult",
        "Senior"
    ],
    include_lowest=True
)

print(
    combined["AgeGroup"]
    .value_counts()
    .sort_index()
)

AgeGroup
Child         102
Teen          147
YoungAdult    568
Adult         397
OlderAdult     85
Senior         10
Name: count, dtype: int64


## 11. Interaction Features

EDA identified an important interaction between passenger sex and passenger class.

Tree-based models can often learn this interaction automatically.

Linear models, however, may benefit from an explicit categorical interaction.

We will therefore create:

`Sex_Pclass`

This combines passenger sex and class into categories such as:

* `male_1`
* `male_3`
* `female_1`
* `female_3`

The original `Sex` and `Pclass` variables will still be retained.

In [26]:
combined["Sex_Pclass"] = (
    combined["Sex"].astype(str)
    + "_"
    + combined["Pclass"].astype(str)
)

print(
    combined["Sex_Pclass"]
    .value_counts()
)

Sex_Pclass
male_3      493
female_3    216
male_1      179
male_2      171
female_1    144
female_2    106
Name: count, dtype: int64


## 12. Review Engineered Features

Before splitting the combined dataset back into training and test sets, we review all newly created features.

This ensures that:

* Every intended feature was successfully created.
* Train and test will receive identical feature definitions.
* No feature was constructed using the target variable.

In [28]:
engineered_features = [
    "Title",
    "FamilySize",
    "IsAlone",
    "FamilyCategory",
    "CabinKnown",
    "Deck",
    "TicketGroupSize",
    "TicketPrefix",
    "LogFare",
    "FarePerPerson",
    "AgeMissing",
    "IsChild",
    "AgeGroup",
    "Sex_Pclass"
]

print(
    f"Total engineered features: "
    f"{len(engineered_features)}"
)

for feature in engineered_features:
    print(
        f"- {feature}"
    )

Total engineered features: 14
- Title
- FamilySize
- IsAlone
- FamilyCategory
- CabinKnown
- Deck
- TicketGroupSize
- TicketPrefix
- LogFare
- FarePerPerson
- AgeMissing
- IsChild
- AgeGroup
- Sex_Pclass


In [29]:
engineered_missing = (
    combined[
        engineered_features
    ]
    .isna()
    .sum()
)

engineered_missing = (
    engineered_missing[
        engineered_missing > 0
    ]
)

if len(engineered_missing) == 0:
    print(
        "No missing values found "
        "in engineered features."
    )
else:
    print(
        "Missing values detected:"
    )

    print(
        engineered_missing
    )

No missing values found in engineered features.


In [30]:
numeric_engineered_features = [
    "FamilySize",
    "IsAlone",
    "CabinKnown",
    "TicketGroupSize",
    "LogFare",
    "FarePerPerson",
    "AgeMissing",
    "IsChild"
]

infinite_counts = {}

for column in numeric_engineered_features:

    infinite_counts[column] = (
        np.isinf(
            combined[column]
        )
        .sum()
    )

infinite_counts

{'FamilySize': np.int64(0),
 'IsAlone': np.int64(0),
 'CabinKnown': np.int64(0),
 'TicketGroupSize': np.int64(0),
 'LogFare': np.int64(0),
 'FarePerPerson': np.int64(0),
 'AgeMissing': np.int64(0),
 'IsChild': np.int64(0)}

## 13. Restore Training and Test Datasets

Feature engineering is now complete.

We will separate the combined predictor dataset back into its original training and test components using the `_Dataset` marker created earlier.

The `Survived` target will then be restored only to the training dataset.

Finally, the temporary dataset-origin marker will be removed.


In [31]:
train_fe = (
    combined[
        combined["_Dataset"] == "train"
    ]
    .drop(
        columns="_Dataset"
    )
    .copy()
)

test_fe = (
    combined[
        combined["_Dataset"] == "test"
    ]
    .drop(
        columns="_Dataset"
    )
    .copy()
)

# Reset indexes
train_fe.reset_index(
    drop=True,
    inplace=True
)

test_fe.reset_index(
    drop=True,
    inplace=True
)

# Restore target
train_fe["Survived"] = (
    y.reset_index(
        drop=True
    )
)

print(
    f"Engineered training shape: "
    f"{train_fe.shape}"
)

print(
    f"Engineered test shape: "
    f"{test_fe.shape}"
)

Engineered training shape: (891, 26)
Engineered test shape: (418, 25)


In [32]:
assert len(train_fe) == len(train)
assert len(test_fe) == len(test)

assert train_fe["PassengerId"].is_unique
assert test_fe["PassengerId"].is_unique

assert "Survived" in train_fe.columns
assert "Survived" not in test_fe.columns

assert train_fe["Survived"].equals(
    train["Survived"]
)

train_predictors = set(
    train_fe.columns
) - {"Survived"}

test_predictors = set(
    test_fe.columns
)

assert train_predictors == test_predictors

print(
    "All structural validation checks passed."
)

All structural validation checks passed.


In [33]:
assert train_fe[
    "PassengerId"
].tolist() == train[
    "PassengerId"
].tolist()

assert test_fe[
    "PassengerId"
].tolist() == test[
    "PassengerId"
].tolist()

print(
    "Passenger order and IDs "
    "were preserved correctly."
)

Passenger order and IDs were preserved correctly.


In [34]:
shape_comparison = pd.DataFrame({
    "Dataset": [
        "Training",
        "Test"
    ],
    "Original Rows": [
        train.shape[0],
        test.shape[0]
    ],
    "Original Columns": [
        train.shape[1],
        test.shape[1]
    ],
    "Engineered Rows": [
        train_fe.shape[0],
        test_fe.shape[0]
    ],
    "Engineered Columns": [
        train_fe.shape[1],
        test_fe.shape[1]
    ]
})

shape_comparison

,Dataset,Original Rows,Original Columns,Engineered Rows,Engineered Columns
0,Training,891,12,891,26
1,Test,418,11,418,25


In [35]:
print(
    "Final Training Columns"
)

print("-" * 50)

for index, column in enumerate(
    train_fe.columns,
    start=1
):
    print(
        f"{index:02d}. {column}"
    )

Final Training Columns
--------------------------------------------------
01. PassengerId
02. Pclass
03. Name
04. Sex
05. Age
06. SibSp
07. Parch
08. Ticket
09. Fare
10. Cabin
11. Embarked
12. Title
13. FamilySize
14. IsAlone
15. FamilyCategory
16. CabinKnown
17. Deck
18. TicketGroupSize
19. TicketPrefix
20. LogFare
21. FarePerPerson
22. AgeMissing
23. IsChild
24. AgeGroup
25. Sex_Pclass
26. Survived


# 14. Feature Engineering Summary

The feature-engineering process transformed the original passenger information into a richer set of candidate predictors.

## Features Intentionally Not Created

Several features considered during initial planning were deliberately excluded.

### `Family_Group_ID`

An arbitrary numerical identifier for each surname or family group has no natural ordinal meaning and could introduce misleading relationships into some models.

### `Family_Role`

This feature would rely on several manually designed rules and may duplicate information already available through `Title`, `Sex`, `Age`, and family features.

### `Socioeconomic_Class`

A manually constructed socioeconomic category based on fixed thresholds would duplicate information from `Pclass`, `Fare`, and `Title` while introducing subjective assumptions.

Instead, we will allow machine learning models to learn relationships between these original and engineered variables.

## Important Principle

The engineered features created in this notebook are **candidate predictors**.

Their inclusion does not guarantee improved model performance.

The next stages will evaluate them using:

* Cross-validation.
* Multiple model families.
* Feature importance.
* Feature ablation.
* Validation accuracy.

Only features that contribute to reliable generalization should be retained in the final competition model.


## 15. Save the Engineered Datasets

The engineered training and test datasets will be saved in the `data/processed/` directory.

These datasets contain both:

* Original passenger information.
* Newly engineered candidate features.

Raw high-cardinality columns such as `Name`, `Ticket`, and `Cabin` are intentionally retained at this stage.

They will be excluded or selected explicitly during model preprocessing rather than permanently deleted here.

This keeps the feature-engineering dataset flexible for future experiments.


In [36]:
processed_dir = Path(
    "/mnt/d/Acelin/Work/Code/titanic-kaggle/data/processed"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

train_fe.to_csv(
    processed_dir /
    "train_featured.csv",
    index=False
)

test_fe.to_csv(
    processed_dir /
    "test_featured.csv",
    index=False
)

print(
    "Engineered datasets saved successfully."
)

print(
    "../data/processed/train_featured.csv"
)

print(
    "../data/processed/test_featured.csv"
)

Engineered datasets saved successfully.
../data/processed/train_featured.csv
../data/processed/test_featured.csv


## 16. Validate Saved Files

Saving a dataset successfully does not guarantee that the resulting file is correct.

We will reload both engineered datasets and verify:

* Dataset dimensions.
* Passenger identifiers.
* Target preservation.
* Train-test predictor consistency.

This final check ensures that the files passed to the modeling stage are structurally valid.


In [38]:
train_check = pd.read_csv(
    "/mnt/d/Acelin/Work/Code/titanic-kaggle/data/processed/train_featured.csv"
)

test_check = pd.read_csv(
    "/mnt/d/Acelin/Work/Code/titanic-kaggle/data/processed/test_featured.csv"
)

assert train_check.shape == train_fe.shape
assert test_check.shape == test_fe.shape

assert train_check[
    "PassengerId"
].tolist() == train[
    "PassengerId"
].tolist()

assert test_check[
    "PassengerId"
].tolist() == test[
    "PassengerId"
].tolist()

assert train_check[
    "Survived"
].equals(
    train["Survived"]
)

print(
    "Saved engineered datasets "
    "validated successfully."
)

Saved engineered datasets validated successfully.


# 17. Phase 4 Summary

In this notebook, we transformed the cleaned Titanic dataset into a richer feature set designed to capture passenger characteristics that may be relevant to survival prediction.

## Name Features

We extracted:

* `Title`

Rare and historically equivalent titles were consolidated to reduce unnecessary categorical complexity.

## Family Features

We created:

* `FamilySize`
* `IsAlone`
* `FamilyCategory`

These features represent passenger family structure more directly than `SibSp` and `Parch` alone.

## Cabin Features

We created:

* `CabinKnown`
* `Deck`

This preserved useful cabin-related information without requiring the model to process raw cabin strings directly.

## Ticket Features

We created:

* `TicketGroupSize`
* `TicketPrefix`

Train and test predictor data were temporarily combined so that shared-ticket counts were calculated consistently across the complete competition dataset.

No target information was included in this process.

## Fare Features

We created:

* `LogFare`
* `FarePerPerson`

These features provide alternative representations of the strongly skewed raw fare variable.

## Age Features

We created:

* `AgeMissing`
* Imputed `Age`
* `IsChild`
* `AgeGroup`

Missing ages were estimated using passenger title and class without using the target variable.

## Interaction Features

We created:

* `Sex_Pclass`

This explicitly represents an interaction identified during exploratory analysis.

## Validation

We verified that:

* No rows were accidentally removed.
* Passenger IDs remained unique.
* Passenger ordering was preserved.
* The target variable remained unchanged.
* Train and test predictor columns remained consistent.
* Engineered features contained no unexpected missing or infinite values.
* Saved datasets could be successfully reloaded.